In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')

Aim: tune the HGBC classification model to maximise balanced accuracy.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn import set_config
set_config(transform_output='pandas')

# pop target labels
y = df.pop('health_condition')

# custom functions for preprocessing
def add_na_count(X):
    X = X.copy()
    X['na_count'] = X.isna().sum(axis=1)
    return X

def add_prod(X):
    X = X.copy()
    X['activity_x_duration'] = (
        X['ordinal_encoding__physical_activity_level']
        * X['numerical__exercise_duration']
    )
    return X

def drop_id(X):
    X = X.copy()
    return X.drop(columns="id")

# separate pipelines for numerical, ordinal, and nominal data
nom_pre = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='other')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cats = [
    ['non-veg', 'balanced', 'veg'],
    ['low', 'medium', 'high'],
    ['poor', 'average', 'good'],
    ['sedentary', 'moderate', 'active'],
    ['no', 'occasional', 'yes']
]

ord_pre = Pipeline([
    ('encode', OrdinalEncoder(categories=cats, handle_unknown='use_encoded_value',
        unknown_value=np.nan, encoded_missing_value=np.nan)),
    ('impute', SimpleImputer(strategy='median'))
])

num_pre = Pipeline([
    ('impute', SimpleImputer(strategy='median'))
])

ordinal = ['diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol']
numeric = ['sleep_duration', 'heart_rate', 'bmi',
       'calorie_expenditure', 'step_count', 'exercise_duration',
       'water_intake']

# combine preprocessing steps into one Column Transformer
encode_impute = ColumnTransformer([
    ('gender_encoding', nom_pre, ['gender']),
    ('ordinal_encoding', ord_pre, ordinal),
    ('numerical', num_pre, numeric)
], remainder='passthrough')

# define classification model
from sklearn.ensemble import HistGradientBoostingClassifier
model = HistGradientBoostingClassifier(random_state=67, class_weight='balanced')

# create classification pipeline
pipe = Pipeline([
    ('drop_id', FunctionTransformer(drop_id, validate=False)),
    ('na_counter', FunctionTransformer(add_na_count, validate=False)),
    ('encode_and_impute', encode_impute),
    ('add_product', FunctionTransformer(add_prod, validate=False)),
    ('classifier', model)
])

In [3]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import uniform

# define search boundaries
param_grid = {
    'classifier__learning_rate': uniform(loc=0.05, scale=0.15),
    'classifier__max_depth': [10, 15, 20],
    'classifier__max_iter': [200, 300, 500]
}

# set up search
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=67)
clf = RandomizedSearchCV(
    pipe,
    param_grid,
    n_iter=10, # close to optimal without taking too long
    scoring="balanced_accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=5,
    random_state=67
)

# search and output best parameters
clf.fit(df, y)
print(clf.best_score_)
print(clf.best_params_)

Fitting 4 folds for each of 10 candidates, totalling 40 fits
0.9370955582424683
{'classifier__learning_rate': np.float64(0.07817570333887214), 'classifier__max_depth': 10, 'classifier__max_iter': 500}


In [4]:
# feed best parameters into model
params = clf.best_params_

model = HistGradientBoostingClassifier(
    random_state=67, 
    learning_rate=params['classifier__learning_rate'], 
    max_depth=params['classifier__max_depth'], 
    max_iter=params['classifier__max_iter'], 
    class_weight='balanced'
)

# create final classification pipeline
pipe = Pipeline([
    ('drop_id', FunctionTransformer(drop_id, validate=False)),
    ('na_counter', FunctionTransformer(add_na_count, validate=False)),
    ('encode_and_impute', encode_impute),
    ('add_product', FunctionTransformer(add_prod, validate=False)),
    ('classifier', model)
])

pipe.fit(df, y)

# make submission
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')
ids = test_df['id']
predictions = pipe.predict(test_df)
submission = pd.DataFrame({'id': ids, 'health_condition': predictions})
submission.to_csv('health_submission.csv', index=False)

In [5]:
# extract preprocessing pipeline
preprocess = Pipeline([
    ('drop_id', FunctionTransformer(drop_id, validate=False)),
    ('na_counter', FunctionTransformer(add_na_count, validate=False)),
    ('encode_and_impute', encode_impute),
    ('add_product', FunctionTransformer(add_prod, validate=False)),
])

X = preprocess.fit_transform(df)

# find permutation importance
from sklearn.inspection import permutation_importance
result = permutation_importance(
    model,
    X,
    y,
    scoring="balanced_accuracy",
    n_repeats=5,
    random_state=67,
    n_jobs=-1
)

perm = pd.Series(
    result.importances_mean,
    index=X.columns
).sort_values(ascending=False)

perm

numerical__sleep_duration                    0.394734
ordinal_encoding__stress_level               0.388705
numerical__bmi                               0.071090
activity_x_duration                          0.062929
remainder__na_count                          0.061296
ordinal_encoding__physical_activity_level    0.014055
ordinal_encoding__smoking_alcohol            0.006537
numerical__calorie_expenditure               0.005754
numerical__step_count                        0.005183
numerical__water_intake                      0.003844
ordinal_encoding__sleep_quality              0.002413
numerical__heart_rate                        0.001466
numerical__exercise_duration                 0.001225
gender_encoding__gender_female               0.001087
gender_encoding__gender_male                 0.000269
gender_encoding__gender_other                0.000206
ordinal_encoding__diet_type                  0.000158
dtype: float64

From permutation importance analysis, sleep duration and stress level are the most important predictors of health risk. 

Engineered features were also informative (>0.06) - combining activity level and duration of exercise gave more predictive value than each of them by themselves.